# Files — Claude Managed Agents

So far the agent only worked with what you typed. Real tasks come with material: documents to review, data to analyze, code to fix. This module moves files **into** the sandbox and gets the agent's work back **out**:

```
files.upload()                      →  file_id, stored in your organization's Files storage
sessions.create(resources=[...])    →  a read-only copy mounted at /mnt/session/uploads/...
sessions.resources.add()            →  mount another file into a session that is already running
agent writes /mnt/session/outputs/  →  each output file shows up in the Files API, scoped to the session
beta.files.list(scope_id=...)       →  find the outputs;  files.download()  →  save them locally
```

The task: review a folder of supplier contracts and produce a summary spreadsheet and a risk report. The contracts live in [`sample-data/contracts/`](../sample-data/contracts/) — fictitious agreements shared by later modules too.

> Module 07 runs this same review as a **dynamic workflow**, with one agent per contract. Here a single agent does it all.

## 1. Setup

In [1]:
%pip install -q -U "anthropic>=1.13" python-dotenv --user


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

client = anthropic.Anthropic()
CONTRACTS_DIR = Path("../sample-data/contracts")
print("Client ready —", len(list(CONTRACTS_DIR.glob("*.md"))), "contracts found")

Client ready — 6 contracts found


## 2. Create an Environment and an Agent

The system prompt tells the agent **where things are**: inputs under `/mnt/session/uploads/`, deliverables under `/mnt/session/outputs/`. Only files written to the outputs directory come back through the API, so the agent needs to know about it.

The agent keeps the default toolset (read, write, bash, …), minus the web tools it doesn't need.

In [3]:
environment = client.beta.environments.create(
    name="files-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
)

agent = client.beta.agents.create(
    name="Contract Analyst",
    model="claude-opus-5-5",
    system=(
        "You are a contract analyst for Brightloom Inc. "
        "Contracts to review are mounted read-only under /mnt/session/uploads/contracts/. "
        "Write every deliverable to /mnt/session/outputs/ — files anywhere else are not returned to the user. "
        "Quote clause numbers when you cite a contract. Keep chat replies short; put the detail in the files."
    ),
    tools=[{
        "type": "agent_toolset_20260401",
        "configs": [
            {"name": "web_search", "enabled": False},
            {"name": "web_fetch", "enabled": False},
        ],
    }],
)

print(f"Environment : {environment.id}")
print(f"Agent       : {agent.id}  (v{agent.version})")

Environment : env_01WFmDeW1GAJXaNGUUE8wNKR
Agent       : agent_01P4ph55EAUEc7sCX8ivKeuK  (v1)


## 3. Upload Files

Uploads go through the **Files API** — the same one the Messages API uses, independent of agents and sessions. Each upload returns a `file_id`. Upload once, then mount the same file into as many sessions as you like.

The filename travels with the file, so give files **descriptive names**: the agent sees them.

We upload five of the six contracts now and keep the last one for later.

In [4]:
contract_paths = sorted(CONTRACTS_DIR.glob("*.md"))
initial, late = contract_paths[:5], contract_paths[5]

uploaded = {}  # filename -> file_id
for path in initial:
    file = client.files.upload(file=path)
    uploaded[path.name] = file.id
    print(f"{file.id}  {file.filename}  ({file.size_bytes} bytes)")

file_014bnHq7yTYC97EXcZAd1Yaw  01-halvard-freight.md  (1514 bytes)


file_01W7PTvL4kdjs2qf65ieoHAr  02-corvel-analytics.md  (1313 bytes)


file_01XxJFvDP6WCnATweo8WBwYj  03-mirefield-packaging.md  (1271 bytes)


file_01VRrjKfCTfUQmFeiHKeiLLh  04-tessaly-cleaning.md  (1022 bytes)


file_01YWymkwS8hZcwb1yo4TZH5u  05-quillon-cloud.md  (1372 bytes)


## 4. Mount Files into a Session

Files reach the sandbox through the session's **`resources`**. Each entry is a `file` with a `file_id` and an optional `mount_path`:

- The `mount_path` is rooted under `/mnt/session/uploads/`, so `/contracts/x.md` lands at `/mnt/session/uploads/contracts/x.md`. Parent directories are created for you.
- Without a `mount_path`, the file lands at `/mnt/session/uploads/<file_id>` — an unhelpful name for the agent.
- Mounted files are **read-only copies**. The agent writes changed versions to new paths. Session copies don't count against your storage.
- Up to 500 files per session.

In [5]:
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    title="Contract review",
    resources=[
        {"type": "file", "file_id": file_id, "mount_path": f"/contracts/{name}"}
        for name, file_id in uploaded.items()
    ],
)

print(f"Session ID: {session.id}")
for resource in client.beta.sessions.resources.list(session.id).data:
    print(f"  {resource.id}  {resource.file_id}  ->  {resource.mount_path}")

Session ID: sesn_01DnF8jQPgTc5RueCXh7a2tH
  sesrsc_013JroGuhiCMRf3oYW2fssJT  file_01LS1eKCtigssaB8aF1BTqoe  ->  /mnt/session/uploads/contracts/01-halvard-freight.md
  sesrsc_01NxfH6uxisVY7P9C6aUtmWD  file_01G2TghkX5kQpgLq4YapLe2m  ->  /mnt/session/uploads/contracts/02-corvel-analytics.md
  sesrsc_01PvvbEcGiP6yc7V2bqZQq4y  file_01WVdyax7J5cvm1WS7FCBiCg  ->  /mnt/session/uploads/contracts/03-mirefield-packaging.md
  sesrsc_01RgRXVm9SX3vn2FQGVtix5L  file_01GvRBkDAFyjQxmpmUC4peAM  ->  /mnt/session/uploads/contracts/04-tessaly-cleaning.md
  sesrsc_01QXcMa1JA18vEbrCnqce6qA  file_01HnYmEeFMcDtjZkU5FACq3M  ->  /mnt/session/uploads/contracts/05-quillon-cloud.md


## 5. Run the Review

The event loop is the one from module 03, trimmed: this agent has no custom tools and its toolset is `always_allow`, so nothing waits on us. It prints each tool call so you can watch the agent work through the files.

In [6]:
def run_turn(session_id: str, text: str) -> None:
    """Send one user message and stream until the turn ends."""
    with client.beta.sessions.events.stream(session_id) as stream:
        client.beta.sessions.events.send(
            session_id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": text}]}],
        )
        for event in stream:
            match event.type:
                case "agent.message":
                    for block in event.content:
                        if block.type == "text":
                            print(f"Agent: {block.text}")
                case "agent.tool_use":
                    detail = event.input.get("command") or event.input.get("file_path") or event.input.get("pattern") or ""
                    print(f"[{event.name}] {str(detail)[:100]}")
                case "session.status_idle":
                    if event.stop_reason.type == "requires_action":
                        continue
                    print(f"[idle: {event.stop_reason.type}]")
                    break
                case "session.status_terminated":
                    print("[terminated]")
                    break

In [7]:
run_turn(
    session.id,
    "Review every contract under /mnt/session/uploads/contracts/. Write two files to /mnt/session/outputs/: "
    "summary.csv with columns supplier,effective_date,auto_renews,non_renewal_notice_days,"
    "termination_for_convenience,liability_cap,change_of_control,governing_law — one row per contract; "
    "and report.md, a one-page risk report that ranks the contracts from most to least risky for Brightloom "
    "and explains the top risks of each.",
)

[bash] cd /mnt/session/uploads/contracts/; ls -laR; wc -c *


[bash] cd /mnt/session/uploads/contracts/; for f in *; do echo "=== $f"; cat "$f"; echo; done


[bash] date


[bash] for d in "2027-01-31 -90 days" "2027-03-15 -30 days" "2027-12-31 -180 days" "2027-03-15 -60 days"; d


[bash] mkdir -p /mnt/session/outputs; cat > /mnt/session/outputs/summary.csv <<'EOF'
supplier,effective_dat


[write] /mnt/session/outputs/report.md


Agent: I reviewed all 5 contracts and wrote both files: the summary CSV with one row per contract, and the one-page risk report.

**Urgent:** the Halvard Freight contract renews automatically for another year unless Brightloom sends written notice by **2 Nov 2026**, which is 22 days from today (cl. 1.2).

**Risk ranking, most to least risky:**
1. **Quillon Cloud – High.** It hosts the order-management system. After the 3-year term it renews for 2 years at a time, and opting out needs 180 days' notice (next deadline 4 Jul 2027, cl. 1.1). Leaving early means paying all fees for the rest of the term (cl. 3.1). Service credits are the only remedy for downtime (cl. 2.2). Liability is capped at 6 months' fees (cl. 4.1), and Quillon can pass the contract to an acquirer without Brightloom's consent (cl. 5.1).
2. **Mirefield Packaging – High-Medium.** Brightloom must buy 250,000 units a year or pay EUR 0.40 per unit short (cl. 2). Brightloom cannot terminate for convenience (cl. 3.2), and dispu

## 6. Add a File to a Running Session

A session's files aren't fixed at creation. **`sessions.resources.add()`** mounts another file into a running session, and **`sessions.resources.delete()`** removes one by its resource `id`.

The session keeps its sandbox between turns — `summary.csv` and `report.md` are still on disk — so the agent can update its work instead of starting over.

In [8]:
late_file = client.files.upload(file=late)
uploaded[late.name] = late_file.id

resource = client.beta.sessions.resources.add(
    session.id,
    type="file",
    file_id=late_file.id,
    mount_path=f"/contracts/{late.name}",
)
print(f"Added {resource.id} -> {resource.mount_path}")

Added sesrsc_01WLPc4RXtJXpertdwU1J7wE -> /mnt/session/uploads/contracts/06-orrin-staffing.md


In [9]:
run_turn(
    session.id,
    f"A new contract was just added: /mnt/session/uploads/contracts/{late.name}. "
    "Review it and update both summary.csv and report.md in /mnt/session/outputs/ to include it.",
)

[bash] cat /mnt/session/uploads/contracts/06-orrin-staffing.md; ls /mnt/session/uploads/contracts/


[bash] date +%F; date -d "2027-09-30 -60 days" +%F


[bash] cat >> /mnt/session/outputs/summary.csv <<'EOF'
Orrin Staffing Partners,2026-04-01,Yes - successive 


Agent: Now the report: add Orrin at rank 5 (Tessaly moves to 6), plus deadline and action rows.
[bash] cd /mnt/session/outputs && python3 - <<'EOF'
p='report.md'; s=open(p).read()
s=s.replace("*Prepared 


[bash] cd /mnt/session/outputs && grep -n "Orrin\|^## \|contracts reviewed" report.md


Agent: I reviewed the Orrin Staffing contract and added it to both files: a new row in summary.csv, and a new section in report.md ranked **5th of 6 (Low-Medium)**, which moves Tessaly to 6th. The other rankings haven't changed.

**Main risks in Orrin:**
- **Staff can be pulled at short notice:** Orrin can end the contract on 30 days' notice (cl. 3.1), so it could withdraw temporary staff just before or during peak season.
- **Narrow indemnity:** Orrin only covers wage-and-hour claims (cl. 4.1). Warehouse injury, discrimination and co-employment claims fall under a mutual USD 1M cap (cl. 4.2), which may be low for a serious warehouse injury.
- **Hiring fee:** taking on an Orrin worker directly within 6 months of their last assignment costs 20% of their first-year salary (cl. 2.2).
- **Mitigants:** Brightloom can also leave on 30 days' notice without a fee (cl. 3.1). If Orrin changes ownership, Brightloom can terminate on 15 days' notice (cl. 5.1).

**Next deadline:** non-renewal notice

## 7. Download the Outputs

Every file the agent writes under `/mnt/session/outputs/` is uploaded to the Files API **scoped to the session**. List them with `beta.files.list(scope_id=session.id)`. Filtering by scope needs the Managed Agents beta header, which `beta.files.list` doesn't add by itself, so we pass it in `betas`.

The session's file list also includes the **session copies of the files you mounted**. Those come back with `downloadable: False`, since only files the session produced can be downloaded, so we skip them.

Outputs can appear **a few seconds after** the session goes idle, so we poll briefly. Should a name be listed more than once, we keep the newest copy.

In [10]:
import time

EXPECTED = {"summary.csv", "report.md"}


def session_outputs(session_id: str, expected: set[str], timeout: float = 60) -> dict:
    """Return {filename: newest file} for the session's outputs, waiting until `expected` are listed."""
    deadline = time.monotonic() + timeout
    while True:
        newest = {}
        for f in client.beta.files.list(scope_id=session_id, betas=["managed-agents-2026-04-01"]):
            if not f.downloadable:   # session copies of the files we mounted
                continue
            if f.filename not in newest or f.created_at > newest[f.filename].created_at:
                newest[f.filename] = f
        if expected <= newest.keys() or time.monotonic() > deadline:
            return newest
        time.sleep(3)


outputs = session_outputs(session.id, EXPECTED)
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
for name, f in outputs.items():
    client.files.download(f.id).write_to_file(out_dir / name)
    print(f"{f.id}  {name}  ({f.size_bytes} bytes)  ->  {out_dir / name}")

file_01RU2sirMpusbXXeRXvK7tws  summary.csv  (2394 bytes)  ->  outputs/summary.csv


file_01QhWjB4rbSbDQ8tVJdySvGC  report.md  (6195 bytes)  ->  outputs/report.md


In [11]:
from IPython.display import Markdown, display

print((out_dir / "summary.csv").read_text())
display(Markdown((out_dir / "report.md").read_text()))

supplier,effective_date,auto_renews,non_renewal_notice_days,termination_for_convenience,liability_cap,change_of_control,governing_law
Halvard Freight Ltd.,2025-02-01,Yes - successive 1-year terms after 2-year initial term (cl. 1.2),90,"Yes - Customer on 60 days' notice + fee of 2 months' average fees (cl. 3.2)","Mutual: fees paid in prior 12 months; excludes confidentiality breaches and indemnities (cl. 4.1-4.2)","No CoC clause; either party may assign without consent to a merger/asset-sale successor (cl. 5.1)",New York
"Corvel Analytics, Inc.",2025-03-15,Yes - successive 1-year terms (cl. 1.1),30,No express right (only CoC exit in cl. 5.1),"Vendor only: fees paid in prior 3 months; mutual exclusion of indirect/consequential damages (cl. 4.1-4.2)","Customer may terminate on 30 days' notice with pro-rata refund if Vendor has a CoC (cl. 5.1)",California
Mirefield Packaging GmbH,2024-06-01,No - fixed 3-year term (cl. 1.1),N/A,"No - expressly excluded (cl. 3.2)","Seller only: EUR 500,000 t

# Supplier Contract Risk Report — Brightloom Inc.
*Prepared 11 Oct 2026 · 6 contracts reviewed · Ranked most → least risky*

## ⚠ Upcoming deadlines
| Deadline | Contract | Action |
|---|---|---|
| **2 Nov 2026** (22 days) | Halvard | Last day for non-renewal notice before auto-renewal to 31 Jan 2028 (cl. 1.2) |
| 14 Jan 2027 | Corvel | Last day Vendor can give notice of a price change for the 15 Mar 2027 renewal (cl. 2.2) |
| 13 Feb 2027 | Corvel | Last day to cancel before the 15 Mar 2027 renewal (cl. 1.1) |
| 31 May 2027 | Mirefield | Contract year 3 and the agreement end; 250k-unit minimum applies to this year (cl. 1.1, 2.1) |
| 4 Jul 2027 | Quillon | Last day for non-renewal notice, or Brightloom is locked in until 31 Dec 2029 (cl. 1.1) |
| 1 Aug 2027 | Orrin | Last day for non-renewal notice before the first renewal on 30 Sep 2027 (cl. 1.1). Either party can also terminate on 30 days' notice at any time (cl. 3.1). |

## 1. Quillon Cloud Services: HIGH
Hosts the order-management system, which is critical to the business, so this has the most exposure.
- **Lock-in:** 3-year term, then automatic **2-year** renewals with **180 days'** notice (cl. 1.1). The only way to leave early is to pay all fees for the rest of the term (cl. 3.1).
- **Weak downtime remedy:** Service credits are capped at 30% of the monthly fee and are the **sole and exclusive remedy** for downtime (cl. 2.1–2.2). Brightloom cannot claim for lost sales or terminate over chronic outages.
- **Low cap:** Fees from the prior 6 months (cl. 4.1). Data-protection breaches are excluded from the cap (cl. 4.2).
- **Change of control:** Provider may assign the agreement to an acquirer without consent, and Brightloom has no exit right (cl. 5.1).

## 2. Mirefield Packaging: HIGH-MEDIUM
- **Take-or-pay:** Brightloom must buy at least 250,000 units a year or pay EUR 0.40 for each unit short, up to about EUR 100k a year (cl. 2.1–2.2). Check volumes for year 3, which ends 31 May 2027.
- **No exit:** Brightloom has no right to terminate for convenience (cl. 3.2). The only exit is for breach, after a 45-day cure period (cl. 3.1).
- **Limited remedies:** Defective goods are only replaced or refunded (cl. 4.1). Total cap is EUR 500k (cl. 4.2).
- **Foreign forum:** German law, with exclusive jurisdiction in the Hamburg courts (cl. 6), so disputes would cost more.
- **Mutual change of control:** If Brightloom has a change of control, it must notify Mirefield within 10 business days, and Mirefield may then terminate (cl. 5.1). This matters for M&A planning.
- *Mitigant:* The contract does not renew automatically and expires 31 May 2027 (cl. 1.1). Start re-sourcing or renegotiation now.

## 3. Corvel Analytics: MEDIUM
- **Uncapped price increases:** The Vendor may raise fees by any amount at renewal on 60 days' notice (cl. 2.2). Combined with automatic renewal and a 30-day cancellation window (cl. 1.1), Brightloom could be renewed at a higher price if it misses the window.
- **Very low cap:** Fees from the prior 3 months, and indirect and consequential damages are excluded (cl. 4.1–4.2). The forecasting data could be affected if something goes wrong.
- **Data:** Customer data is deleted 30 days after termination unless Brightloom requests an export (cl. 3.2). The Vendor may use de-identified usage data (cl. 3.1).
- **No termination for convenience.** *Mitigant:* Brightloom can exit with a pro-rata refund if the Vendor has a change of control (cl. 5.1).

## 4. Halvard Freight: MEDIUM-LOW
- **Renewal deadline now:** The agreement auto-renews for 1 year unless notice is given by **2 Nov 2026** (cl. 1.2).
- **Termination for convenience costs money:** 60 days' notice plus a fee of 2 months' average fees (cl. 3.2).
- **Uncapped mutual exposure:** Confidentiality breaches and indemnities are excluded from the cap (cl. 4.2). This also applies to Brightloom.
- **Change of control:** Either party may assign to a merger successor without consent (cl. 5.1), so Brightloom could end up with a new counterparty.
- *Mitigants:* Price rises are limited to CPI (cl. 2.2). The cap is reasonable at 12 months' fees and is mutual (cl. 4.1). Payment terms are net 45 (cl. 2.1).

## 5. Orrin Staffing: LOW-MEDIUM
- **Supply risk at peak:** Orrin can terminate for convenience on 30 days' notice (cl. 3.1), so it could withdraw temporary staff just before or during peak season.
- **Narrow indemnity:** Orrin only indemnifies wage-and-hour claims (cl. 4.1). Warehouse injuries, discrimination and joint-employer claims fall under the mutual USD 1M cap (cl. 4.2), which may be low for a serious warehouse injury.
- **Costs:** Overtime is billed at 1.5× (cl. 2.1). Hiring an Orrin worker directly within 6 months costs 20% of their first-year salary (cl. 2.2).
- *Mitigants:* Brightloom can also exit on 30 days' notice without a fee (cl. 3.1). If Orrin has a change of control, it must notify Brightloom, which can then terminate on 15 days' notice (cl. 5.1).

## 6. Tessaly Cleaning: LOW
- Month-to-month since 1 Sep 2026. Either party can terminate on 30 days' notice without penalty (cl. 1.1, 3.1).
- Contractor liability for damage to property is **uncapped** (cl. 4.2) and backed by USD 2M general liability insurance (cl. 4.1). Staff are background-checked (cl. 5.1).
- **Residual risk:** Tessaly can also leave on 30 days' notice, so cleaning cover could stop at short notice. There is no indemnity or confidentiality clause for after-hours access to HQ. The fee is fixed at USD 6,800/month (cl. 2.1).

## Recommended actions
1. **Halvard:** Decide by 2 Nov 2026 whether to renew. If not, send written notice.
2. **Quillon:** Negotiate a termination right for chronic SLA failure, a higher cap, and consent or an exit right on assignment. Diarise 4 Jul 2027.
3. **Mirefield:** Forecast year-3 volume against the 250k minimum and start re-sourcing for June 2027.
4. **Corvel:** Before 14 Jan 2027, seek a cap on price increases. Diarise 13 Feb 2027 and export data before any exit.
5. **Orrin:** Ask for a longer notice period or a peak-season lock on Orrin's termination right, wider indemnities (injury, discrimination, co-employment), and evidence of workers' comp insurance.


## 8. Clean Up

Uploaded files stay in your organization's storage until you delete them. The session's mounted copies and its output files don't count against your storage limit, but the originals do. Delete the uploads when you no longer need them — the session keeps its own copies, so this doesn't break it.

In [12]:
for name, file_id in uploaded.items():
    client.files.delete(file_id)
    print(f"Deleted {file_id}  {name}")

Deleted file_014bnHq7yTYC97EXcZAd1Yaw  01-halvard-freight.md


Deleted file_01W7PTvL4kdjs2qf65ieoHAr  02-corvel-analytics.md


Deleted file_01XxJFvDP6WCnATweo8WBwYj  03-mirefield-packaging.md


Deleted file_01VRrjKfCTfUQmFeiHKeiLLh  04-tessaly-cleaning.md


Deleted file_01YWymkwS8hZcwb1yo4TZH5u  05-quillon-cloud.md


Deleted file_013uhWyBme72LqwmTRMnLYyM  06-orrin-staffing.md


## Summary

```
files.upload(file=path)                      →  file_id (descriptive filename!)
sessions.create(resources=[{type: file,      →  read-only at /mnt/session/uploads/<mount_path>
    file_id, mount_path}])
sessions.resources.add / list / delete       →  change a running session's files
agent writes /mnt/session/outputs/...        →  uploaded to the Files API, scoped to the session
beta.files.list(scope_id=session.id, betas=[...])  →  keep downloadable ones, files.download(id)
```

Key takeaways:
- Upload once, mount into any number of sessions; the filename is what the agent sees
- Mounted files are read-only; tell the agent where inputs live and where outputs go
- Only `/mnt/session/outputs/` comes back — and it can lag the idle by a few seconds
- The sandbox persists between turns, so later turns build on earlier files

### Next steps
- **05-session-control** — steer, interrupt, reconnect, and cap spend with a session budget
- **06-subagents** — delegate to other agents in session threads
- **07-dynamic-workflows** — let the agent write a workflow that runs many agents in parallel